In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/K2Debug/Financial-News-Sentiment-Analyzer-for-Economic-Forecasting.git"
PROJECT_ROOT = Path("/content/EF-02")
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not PROJECT_ROOT.exists():
        print("Workspace not found — cloning repo and installing dependencies...")
        !git clone {REPO_URL} {PROJECT_ROOT}
        !pip install -q python-dotenv groq openai bs4
        !pip install -q -r /content/EF-02/requirements.txt
        print("Workspace ready.")
    os.chdir(NOTEBOOKS_DIR)

# 04: Data Consolidation

Aggregates exchange rate, CPI, and classified headline data to monthly level and merges into a single dataset for visualisation.

In [1]:
import pandas as pd

## Step 1: Exchange Rate

Source: `usd_tzs_2022_2024.csv`  daily USD/TZS data from Investing.com. Price column contains comma-formatted strings and must be cleaned before numeric conversion. Resampled to monthly average with a month-on-month percentage change column added.

In [2]:
df_exc = pd.read_csv("../data/raw/usd_tzs_2022_2024.csv")
df_exc.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,12/31/2024,"2,425.00","2,435.00","2,435.00","2,405.00",NaN,-0.82%
1,12/30/2024,"2,445.00","2,415.00","2,470.00","2,415.00",NaN,0.82%
2,12/27/2024,"2,425.00","2,420.00","2,432.50","2,407.50",NaN,1.46%
3,12/26/2024,"2,390.00","2,390.00","2,390.00","2,390.00",NaN,0.00%
4,12/25/2024,"2,390.00","2,390.00","2,390.00","2,390.00",NaN,-1.24%


In [3]:
# Clean comma-formatted price strings and convert to numeric
df_exc["Price"] = (
    df_exc["Price"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df_exc["Price"] = pd.to_numeric(df_exc["Price"], errors="coerce")

# Parse dates and keep only Date and Price
df_exc["Date"] = pd.to_datetime(df_exc["Date"], format="%m/%d/%Y")
df_exc = df_exc[["Date", "Price"]]

# Resample to monthly average and compute month-on-month change
df_exc_monthly = (
    df_exc.set_index("Date")
    .resample("ME")["Price"]
    .mean()
    .reset_index()
)

df_exc_monthly["YearMonth"] = df_exc_monthly["Date"].dt.strftime("%Y-%m")
df_exc_monthly["Rate_Change_%"] = (
    df_exc_monthly["Price"].pct_change(fill_method=None) * 100
)
df_exc_monthly = (
    df_exc_monthly[["YearMonth", "Price", "Rate_Change_%"]]
    .rename(columns={"Price": "USD/TZS_Rate"})
)
df_exc_monthly["Rate_Change_%"] = df_exc_monthly["Rate_Change_%"].fillna(0)
df_exc_monthly.head()

,YearMonth,USD/TZS_Rate,Rate_Change_%
0,2022-01,2303.380952,0.000000
1,2022-02,2308.800000,0.235265
2,2022-03,2312.565217,0.163081
3,2022-04,2317.761905,0.224715
4,2022-05,2320.818182,0.131863


## Step 2: CPI

Source: `tanzania_cpi_2022_2024.csv` — already at monthly frequency. Only `inflation_rate_%` is carried forward; the index column is dropped.

In [4]:
df_cpi = pd.read_csv("../data/raw/tanzania_cpi_2022_2024.csv")
df_cpi.head()

,date,inflation_rate_pct,all_items_index
0,2022-01-01,3.997044,105.591638
1,2022-02-01,3.673581,106.201764
2,2022-03-01,3.552367,107.085012
3,2022-04-01,3.783246,107.877973
4,2022-05-01,4.026812,108.420516


In [ ]:
df_cpi["date"] = pd.to_datetime(df_cpi["date"], format="%Y-%m-%d")
df_cpi_monthly = df_cpi.copy()
df_cpi_monthly["YearMonth"] = df_cpi_monthly["date"].dt.strftime("%Y-%m")
df_cpi_monthly = df_cpi_monthly.drop(columns=["date", "all_items_index"])
df_cpi_monthly = df_cpi_monthly.rename(columns={"inflation_rate_pct": "inflation_rate_%"})

# FIX: CPI must cover Jan 2022 – Dec 2024 (36 months). The inner join in Step 4
# drops any month missing from ANY source — Dec 2024 was lost when this file
# stopped at Nov 2024. Ensure tanzania_cpi_2022_2024.csv includes 2024-12.
EXPECTED_MONTHS = 36
print(f"CPI months loaded: {len(df_cpi_monthly)} (expected {EXPECTED_MONTHS})")
if len(df_cpi_monthly) < EXPECTED_MONTHS:
    print("WARNING: CPI is missing months — merged output will be < 36 rows")

df_cpi_monthly.head()

,inflation_rate_%,YearMonth
0,3.997044,2022-01
1,3.673581,2022-02
2,3.552367,2022-03
3,3.783246,2022-04
4,4.026812,2022-05


## Step 3: Headlines

Source: `tz_headlines_labelled.csv` output of the production classifier. Irrelevant headlines are dropped first. Records are then aggregated to monthly level: total headline count, dominant category by share, and sentiment distribution percentages.

In [6]:
df_hdl = pd.read_csv("../data/processed/tz_headlines_labelled.csv")

print(len(df_hdl), ": total headlines before dropping irrelevant")
df_hdl = df_hdl[df_hdl["relevant"] == True]
print(len(df_hdl), ": total headlines after dropping irrelevant")

6235 : total headlines before dropping irrelevant
5163 : total headlines after dropping irrelevant


In [7]:
df_hdl_clean = df_hdl[["date", "headline", "category", "sentiment"]].copy()

df_hdl_clean["date"] = pd.to_datetime(df_hdl_clean["date"], format="%Y-%m-%d", errors="coerce")
df_hdl_clean["YearMonth"] = df_hdl_clean["date"].dt.strftime("%Y-%m")
df_hdl_clean["category"] = df_hdl_clean["category"].astype("category")
df_hdl_clean["sentiment"] = df_hdl_clean["sentiment"].astype("category")
df_hdl_clean = df_hdl_clean.drop(columns=["headline", "date"])
df_hdl_clean.head()

,category,sentiment,YearMonth
0,Policy,Neutral,2024-12
1,Trade,Neutral,2024-12
2,Investment,Neutral,2024-12
3,Investment,Neutral,2024-12
4,Banking,Positive,2024-12


In [8]:
# Total headlines per month
headline_counts = (
    df_hdl_clean.groupby("YearMonth", observed=True)
    .size()
    .reset_index(name="num_headlines")
)

# Dominant category per month by percentage share
top_category = (
    df_hdl_clean.groupby(["YearMonth", "category"], observed=True)
    .size()
    .reset_index(name="count")
)
top_category["category %"] = (
    top_category.groupby("YearMonth")["count"]
    .transform(lambda x: x / x.sum() * 100)
).round(2)
top_category = (
    top_category.sort_values(["YearMonth", "category %"], ascending=[True, False])
    .groupby("YearMonth")
    .first()
    .reset_index()
    .drop(columns=["count"])
)

# Sentiment distribution per month as percentages
sentiment_counts = (
    df_hdl_clean.groupby(["YearMonth", "sentiment"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
sentiment_cols = [c for c in sentiment_counts.columns if c != "YearMonth"]
sentiment_counts[sentiment_cols] = (
    sentiment_counts[sentiment_cols]
    .div(sentiment_counts[sentiment_cols].sum(axis=1), axis=0) * 100
).round(2)
sentiment_counts["Avg_sentiment"] = sentiment_counts[sentiment_cols].idxmax(axis=1)

# Merge headline aggregates
df_hdl_monthly = (
    headline_counts
    .merge(top_category, on="YearMonth")
    .merge(sentiment_counts, on="YearMonth")
)
df_hdl_monthly.head()

,YearMonth,num_headlines,category,category %,Negative,Neutral,Positive,Avg_sentiment
0,2022-01,56,Trade,28.57,23.08,28.21,48.72,Positive
1,2022-02,43,Trade,32.56,5.26,71.05,23.68,Neutral
2,2022-03,55,Policy,25.45,8.00,26.00,66.00,Positive
3,2022-04,50,Trade,20.00,20.00,42.00,38.00,Neutral
4,2022-05,55,Trade,32.73,9.80,43.14,47.06,Positive


## Step 4: Consolidation

Merges all three monthly DataFrames on `YearMonth` using an inner join to keep only months present across all three sources. Columns are renamed for consistency.

In [9]:
# FIX: inner join keeps only months present in ALL three sources.
# Dec 2024 headlines and FX existed but CPI ended at Nov 2024, so 2024-12 was dropped.
df_merge = pd.merge(df_hdl_monthly, df_cpi_monthly, on="YearMonth", how="inner")
df_merge = pd.merge(df_merge, df_exc_monthly, on="YearMonth", how="inner")

# FIX: sanity check — expect 36 rows (Jan 2022 – Dec 2024)
hdl_ym = set(df_hdl_monthly["YearMonth"])
cpi_ym = set(df_cpi_monthly["YearMonth"])
exc_ym = set(df_exc_monthly["YearMonth"])
dropped = (hdl_ym | cpi_ym | exc_ym) - set(df_merge["YearMonth"])
if dropped:
    for m in sorted(dropped):
        missing_in = []
        if m not in hdl_ym:
            missing_in.append("headlines")
        if m not in cpi_ym:
            missing_in.append("CPI")
        if m not in exc_ym:
            missing_in.append("FX")
        print(f"  {m}: missing in {', '.join(missing_in)}")
print(f"Merged rows: {len(df_merge)} (expected 36)")

df_merge = df_merge.rename(
    columns={
        "category": "Top_Category",
        "Negative": "Negative %",
        "Neutral": "Neutral %",
        "Positive": "Positive %",
        "Avg_sentiment": "Dominant_Sentiment",
        "inflation_rate_%": "Inflation %",
    }
)

df_merge

,YearMonth,num_headlines,Top_Category,category %,Negative %,Neutral %,Positive %,Dominant_Sentiment,Inflation %,USD/TZS_Rate,Rate_Change_%
0,2022-01,56,Trade,28.57,23.08,28.21,48.72,Positive,3.997044,2303.380952,0.000000
1,2022-02,43,Trade,32.56,5.26,71.05,23.68,Neutral,3.673581,2308.800000,0.235265
2,2022-03,55,Policy,25.45,8.00,26.00,66.00,Positive,3.552367,2312.565217,0.163081
3,2022-04,50,Trade,20.00,20.00,42.00,38.00,Neutral,3.783246,2317.761905,0.224715
4,2022-05,55,Trade,32.73,9.80,43.14,47.06,Positive,4.026812,2320.818182,0.131863
5,2022-06,65,Policy,23.08,14.29,42.86,42.86,Neutral,4.436698,2326.090909,0.227193
6,2022-07,70,Trade,24.29,5.17,56.90,37.93,Neutral,4.537499,2327.000000,0.039082
7,2022-08,121,Policy,23.14,8.04,50.89,41.07,Neutral,4.649161,2326.695652,-0.013079
8,2022-09,232,Policy,20.26,7.98,48.83,43.19,Neutral,4.839360,2326.318182,-0.016223
9,2022-10,211,Agriculture,25.12,7.96,44.28,47.76,Positive,4.942250,2326.523810,0.008839


## Step 5: Export

In [13]:
df_merge.to_csv("../data/processed/Visualization_Data.csv", index=False)
print(f"Exported {len(df_merge)} rows to data/processed/Visualization_Data.csv")

Exported 33 rows to data/processed/Visualization_Data.csv
